In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from datasets import load_dataset

device = "cuda" if torch.cuda.is_available() else "cpu"


c:\Users\mittall\source\EAISI\NHS\BPMNLanguageModel\bpmn_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:

model = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"
model_path = "./tinyllama-lora/checkpoint-370"

In [4]:
tokenizer = AutoTokenizer.from_pretrained(model)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(model, torch_dtype=torch.float32, low_cpu_mem_usage=True)
non_instruction_model = PeftModel.from_pretrained(base_model, model_path)
non_instruction_model = non_instruction_model.merge_and_unload()

# PEFT stores state inside layer modules even after merge_and_unload(),
# so del peft_config is not enough. Save + reload as plain weights — the only
# guaranteed way to get a clean model with no PEFT internals.
MERGED_PATH = "./tinyllama-merged-temp"
non_instruction_model.save_pretrained(MERGED_PATH)
non_instruction_model = AutoModelForCausalLM.from_pretrained(
    MERGED_PATH, torch_dtype=torch.float32, low_cpu_mem_usage=True
)
non_instruction_model = non_instruction_model.to(device)
print("Loaded clean merged model, type:", type(non_instruction_model).__name__)


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 2029.70it/s]

Loaded clean merged model, type: LlamaForCausalLM


Test Previous Model

In [5]:

prompt = "The Loop Activity is a type of Activity that "

inputs = tokenizer(prompt, return_tensors="pt").to(device)

outputs = non_instruction_model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1,
)

print("\nModel Output:\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Model Output:

The Loop Activity is a type of Activity that 251 Business Process Model and Notation, v2.0 A Loop Activity can be either a Recurrence Loop or a Repetition Loop. A Recurrence Loop represents a sequence where each occurrence (i.e., each time) of the activity will run. A Repetition Loop represents an infinite repetition of the activity, i.e., the activity runs continuously. A Loop Activity does not have any inherent timing constraints. For more information on how to define the behavior


In [6]:
dataset = load_dataset("json", data_files="data/bpmn_instruction_dataset.jsonl", split="train")
dataset = dataset.select([0, 16])   # 2 examples: basic BPMN + gateway
print("Training questions:")
for ex in dataset:
    print(" -", ex["instruction"])


Training questions:
 - What is BPMN and what is its primary goal?
 - Describe all five Gateway types in BPMN 2.0 and when to use each.


In [7]:
dataset['instruction'][0]

'What is BPMN and what is its primary goal?'

In [8]:
def format_example(example):
    prompt = f"### Instruction:\n{example['instruction']}\n### Input:\n{example['input']}\n### Response:\n{example['output']}"
    return {"text": prompt}

In [9]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [10]:
def tokenize_and_mask(example):
    text = example["text"]

    # Include the newline after ### Response: so the template itself is masked
    response_marker = "### Response:\n"
    response_start = text.find(response_marker)

    # Tokenize full sequence with padding
    enc = tokenizer(text, truncation=True, padding="max_length", max_length=512)
    input_ids = enc["input_ids"]

    if response_start != -1:
        prompt_prefix = text[:response_start + len(response_marker)]
        response_token_start = len(tokenizer(prompt_prefix, truncation=True, max_length=512)["input_ids"])
    else:
        response_token_start = 0

    # Actual (unpadded) sequence length — used to mask trailing padding
    # CRITICAL: pad_token == eos_token, so we MUST NOT label trailing pad positions
    # as valid targets or the model trains to predict EOS everywhere → blank output
    actual_len = len(tokenizer(text, truncation=True, max_length=512)["input_ids"])

    labels = input_ids.copy()
    # Mask prompt + ### Response:\n prefix
    for i in range(min(response_token_start, 512)):
        labels[i] = -100
    # Mask trailing padding (positions beyond actual sequence end)
    for i in range(actual_len, 512):
        labels[i] = -100

    enc["labels"] = labels
    return enc


In [11]:
dataset = dataset.map(format_example)
dataset[0]

Map: 100%|██████████| 2/2 [00:00<00:00, 40.64 examples/s]


{'instruction': 'What is BPMN and what is its primary goal?',
 'input': '',
 'output': 'BPMN stands for Business Process Model and Notation. It is an OMG standard (version 2.0) for graphically representing business processes. Its primary goal is to provide a notation that is readily understandable by all business users—from business analysts who create initial process drafts, to technical developers who implement those processes, to business managers who monitor them. BPMN creates a standardized bridge between business process design and process implementation. A secondary but equally important goal is to ensure that XML-based execution languages, such as WS-BPEL (Web Services Business Process Execution Language), can be visualized in a user-friendly notation.',
 'text': '### Instruction:\nWhat is BPMN and what is its primary goal?\n### Input:\n\n### Response:\nBPMN stands for Business Process Model and Notation. It is an OMG standard (version 2.0) for graphically representing business

In [12]:
# Apply tokenization
tokenized = dataset.map(tokenize_and_mask, batched=False)
print("Tokenization + masking done.")

Map: 100%|██████████| 2/2 [00:00<00:00, 35.20 examples/s]

Tokenization + masking done.


In [13]:
# Verify labels are correct before training
sample = tokenized[0]
labels = sample["labels"]
valid = [i for i, l in enumerate(labels) if l != -100]
print(f"Total tokens: 512 | Prompt masked: {valid[0]} tokens | Response tokens with valid labels: {len(valid)} | Trailing masked: {512 - valid[-1] - 1}")
print("First 5 response token labels (should be real token IDs, not -100):", labels[valid[0]:valid[0]+5])
print("Decoded response (first 80 chars):", tokenizer.decode([l for l in labels if l != -100])[:80])


Total tokens: 512 | Prompt masked: 30 tokens | Response tokens with valid labels: 140 | Trailing masked: 342
First 5 response token labels (should be real token IDs, not -100): [29933, 13427, 29940, 15028, 363]
Decoded response (first 80 chars): BPMN stands for Business Process Model and Notation. It is an OMG standard (vers


In [14]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,                        # was 8 — more rank = more expressiveness per step
    lora_alpha=32,               # keep alpha = 2*r
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # was only q+v; adding k+o doubles trainable params
    bias="none",
)


In [15]:
instruction_model = get_peft_model(non_instruction_model, lora_config)
instruction_model.print_trainable_parameters()
# Should show ~4.5M trainable params and NO "second time" warning


trainable params: 4,505,600 || all params: 1,104,553,984 || trainable%: 0.4079


In [16]:
args = TrainingArguments(
    output_dir="./tinyllama-instruction",
    num_train_epochs=50,             # 50 × 2 examples ÷ batch 1 = 100 steps
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    learning_rate=5e-4,
    fp16=torch.cuda.is_available(),
    logging_steps=20,
    save_total_limit=1,
    report_to="none",
)


In [17]:
trainer = Trainer(
    model=instruction_model     ,
    args=args,
    train_dataset=tokenized,
)

In [18]:
trainer.train()

c:\Users\mittall\source\EAISI\NHS\BPMNLanguageModel\bpmn_env\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
20,0.684883
40,0.007259
60,0.000905
80,0.000544
100,0.000458


TrainOutput(global_step=100, training_loss=0.13880993902683259, metrics={'train_runtime': 1895.6601, 'train_samples_per_second': 0.053, 'train_steps_per_second': 0.053, 'total_flos': 319186324684800.0, 'train_loss': 0.13880993902683259, 'epoch': 50.0})

In [20]:
# Save the model — safe_serialization=False avoids the Windows mmap conflict (os error 1224)
instruction_model.save_pretrained("./tinyllama-instruction", safe_serialization=False)
tokenizer.save_pretrained("./tinyllama-instruction")


('./tinyllama-instruction\\tokenizer_config.json',
 './tinyllama-instruction\\tokenizer.json')

In [22]:
instruction_model.eval()

# Use a question that IS in the training data (idx 16).
# With only ~15 optimizer steps the model can only recall training examples — it
# cannot generalize to paraphrases like "What is gateway?" yet.
prompt = "### Instruction:\nDescribe all five Gateway types in BPMN 2.0 and when to use each.\n### Input:\n\n### Response:\n"
inputs = tokenizer(prompt, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = instruction_model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False,          # greedy — avoids randomly sampling EOS on a weakly-trained model
        repetition_penalty=1.2,
    )

new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
print("\nModel Output:\n")
print(tokenizer.decode(new_tokens, skip_special_tokens=True))



Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Model Output:

BPMN 2.0 defines five Gateway types, all depicted as diamonds:

1. Exclusive Gateway (XOR) – Only one outgoing path is taken based on conditions. At merge, the first arriving token passes through immediately. Icon: X inside diamond. Use when exactly one of several paths should execute.

2. Inclusive Gateway (OR) – One or more outgoing paths are taken based on conditions. At merge, waits for all active incoming tokens to arrive before continuing. Icon: circle inside diamond. Use when any combination of paths may be active.

3. Parallel Gateway (AND) – All outgoing paths are always taken (fork). At merge (join), waits for all incoming paths to arrive before proceeding. Icon: + inside diamond. Use for unconditional parallel execution.

4. Event-Based Gateway – Routes


In [23]:
questions = [
    "What is BPMN and what is its primary goal?",                         # idx 0 — in training
    "Describe all five Gateway types in BPMN 2.0 and when to use each.", # idx 16 — in training
]


In [24]:
for q in questions:
    print("Question:", q)

    print("\n--- Non-instruction model ---")
    inputs = tokenizer(q, return_tensors="pt").to(device)
    outputs = non_instruction_model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=False,
        repetition_penalty=1.3,
    )
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    print(tokenizer.decode(new_tokens, skip_special_tokens=True))

    print("\n--- Instruction-tuned model ---")
    prompt = f"### Instruction:\n{q}\n### Input:\n\n### Response:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    outputs = instruction_model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=False,
        repetition_penalty=1.3,
    )
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    print(tokenizer.decode(new_tokens, skip_special_tokens=True))

    print("=" * 80, "\n")


Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: What is BPMN and what is its primary goal?

--- Non-instruction model ---


Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A. The Business Process Model and Notation (BPMN), released in 2013, is a standard for graphically representing business processes. Its primary goal is to provide a notation that is readily understandable by all business users—from business analysts who create initial process drafts, to technical developers who implement those processes, to business managers who monitor them. Bpm

--- Instruction-tuned model ---


Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


BPMN stands for Business Process Model and Notation. It is an OMG standard (version 2.0) for graphically representing business processes. Its primary goal is to provide a notation that is readily understandable by all business users—from business analysts who create initial process drafts, to technical developers who implement those processes, to business managers who monitor them. BPMN creates a standardized bridge between business process design and process implementation. A secondary but equally important goal is to ensure that XML-based execution languages, such as WS-BPEL (Web Services Business Process Execution Language), can be visualized in a user-friendly notation. A final but lesser goal is to establish a

Question: Describe all five Gateway types in BPMN 2.0 and when to use each.

--- Non-instruction model ---


Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A Gateway is a execution path that waits for conditions to be met before continuing, or activates behavior based on those conditions. Each type of Gateway is identified by a icon in the XML graph representation (Figure 13.4). At first blush, Bridge appears to be the most complex Gateword ever created—with 65 different patterns! In fact, it

--- Instruction-tuned model ---
BPMN 2.0 defines five Gateway types, all depicted as diamonds:

1. Exclusive Gateway (XOR) – Only one outgoing path is taken based on conditions. At merge, the first arriving token passes through immediately. Icon: X inside diamond. Use when exactly one of several paths should execute.

2. Inclusive Gateway (OR) – One or more outgoing paths are taken based on conditions. At merge, waits for all active incoming tokens to arrive before continuing. Icon: circle inside diamond. Use when any combination of paths may be active.

3. Parallel Gateway (AND) – All outgoing paths are always

